# Track A — EDA (Fase 0 + Fase 1)
**BDC Satria Data 2026 — Klasifikasi Citra Sampah**

Notebook ini mencakup:
- Fase 0: Verifikasi dataset (jumlah, struktur)
- Fase 1: EDA — distribusi kelas, statistik gambar, cek corrupt, cek duplikat

> **Output utama:** `eda_stats.json`, `skip_list.txt`
> Output ini jadi input untuk `02_split_verify.ipynb`

## 0. Setup & Mount Drive

In [ ]:
# Install dependencies (jalankan sekali)
!pip install -q imagehash pandas numpy matplotlib seaborn pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError

# ─── KONFIGURASI ──────────────────────────────────────────────────────────────
DATA_ROOT   = Path('/content/drive/MyDrive/BDC2026')
TRAIN_DIR   = DATA_ROOT / 'train'
TEST_DIR    = DATA_ROOT / 'test'
SUB_CSV     = DATA_ROOT / 'submission.csv'
OUTPUT_DIR  = Path('/content/track_a_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Jumlah yang diharapkan
EXPECTED_TRAIN = 26527
EXPECTED_TEST  = 1458

LABEL_MAP = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ['Recyclable', 'Electronic', 'Organic']

print(f'DATA_ROOT: {DATA_ROOT}')
print(f'TRAIN_DIR ada: {TRAIN_DIR.exists()}')
print(f'TEST_DIR  ada: {TEST_DIR.exists()}')

## Fase 0 — Verifikasi Jumlah & Struktur Dataset

In [ ]:
# Scan semua gambar train
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
records = []
for folder_name, label_idx in LABEL_MAP.items():
    class_dir = TRAIN_DIR / folder_name
    assert class_dir.exists(), f'Folder tidak ditemukan: {class_dir}'
    for fp in class_dir.iterdir():
        if fp.suffix.lower() in IMG_EXTS:
            records.append({'filepath': str(fp), 'label_name': folder_name, 'label': label_idx})

df = pd.DataFrame(records)
print(f'Total gambar train: {len(df)}')
print(df['label_name'].value_counts())

# Verifikasi jumlah
assert len(df) == EXPECTED_TRAIN, \
    f'JUMLAH TIDAK COCOK! Dapat {len(df)}, harapkan {EXPECTED_TRAIN}. Cek dataset!'
print(f'\nJumlah train terverifikasi: {len(df)}')

In [ ]:
# Scan gambar test
test_files = [fp for fp in sorted(TEST_DIR.iterdir()) if fp.suffix.lower() in IMG_EXTS]
print(f'Total gambar test: {len(test_files)}')
assert len(test_files) == EXPECTED_TEST, \
    f'JUMLAH TEST TIDAK COCOK! Dapat {len(test_files)}, harapkan {EXPECTED_TEST}'
print('Jumlah test terverifikasi')

## Fase 1A — Distribusi Kelas

In [ ]:
class_counts = df['label_name'].value_counts().sort_index()
print('Distribusi kelas:')
print(class_counts)
print()

# Rasio
total = len(df)
for cls, cnt in class_counts.items():
    print(f'  {cls}: {cnt} ({cnt/total*100:.1f}%)')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
class_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71','#3498db','#e67e22'], edgecolor='black')
axes[0].set_title('Jumlah Gambar per Kelas', fontweight='bold')
axes[0].set_xlabel('Kelas')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(rotation=15)

axes[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71','#3498db','#e67e22'])
axes[1].set_title('Proporsi Kelas', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Catat ke stats
eda_stats = {
    'class_counts': class_counts.to_dict(),
    'class_ratios': (class_counts / total).to_dict(),
}

## Fase 1B — Statistik Ukuran Gambar

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

SAMPLE_N    = 2000   # atau None untuk semua gambar (tetap cepat dengan threading)
MAX_WORKERS = 16

sample_df = df.sample(min(SAMPLE_N, len(df)), random_state=42) if SAMPLE_N else df

def get_stats_one(args):
    idx, fp = args
    try:
        with Image.open(fp) as img:
            w, h = img.size          # ← hanya baca header, TIDAK load seluruh pixel
            c = len(img.getbands())
        return idx, w, h, c, round(w/h, 3)
    except Exception:
        return idx, None, None, None, None

results = [None] * len(sample_df)   # pre-allocate untuk jaga urutan
filepaths = sample_df['filepath'].tolist()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(get_stats_one, (idx, fp)): idx
               for idx, fp in enumerate(filepaths)}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Statistik gambar'):
        idx, w, h, c, ar = future.result()
        results[idx] = (w, h, c, ar)

# Unpack hasil — filter yang None (gagal dibuka)
widths, heights, channels, aspect_ratios = [], [], [], []
for r in results:
    if r and r[0] is not None:
        w, h, c, ar = r
        widths.append(w); heights.append(h)
        channels.append(c); aspect_ratios.append(ar)

print(f'Sampel dianalisis: {len(widths)}')
print(f'Width  — min: {min(widths)}, max: {max(widths)}, median: {int(np.median(widths))}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, median: {int(np.median(heights))}')
print(f'Channels (unik): {set(channels)}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(widths, bins=40, color='#3498db', edgecolor='white')
axes[0].set_title('Distribusi Width'); axes[0].set_xlabel('pixel')
axes[1].hist(heights, bins=40, color='#2ecc71', edgecolor='white')
axes[1].set_title('Distribusi Height'); axes[1].set_xlabel('pixel')
axes[2].hist(aspect_ratios, bins=40, color='#e67e22', edgecolor='white')
axes[2].set_title('Distribusi Aspect Ratio'); axes[2].set_xlabel('W/H')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'image_size_dist.png', dpi=150, bbox_inches='tight')
plt.show()

eda_stats['image_stats'] = {
    'sample_n': len(widths),
    'width':  {'min': min(widths), 'max': max(widths), 'median': int(np.median(widths))},
    'height': {'min': min(heights), 'max': max(heights), 'median': int(np.median(heights))},
    'channels_unique': list(set(channels)),
}


## Fase 1C — Cek Gambar Corrupt

In [ ]:
# ─── Fase 1C — Cek Corrupt (VERSI CEPAT dengan ThreadPoolExecutor) ─────────
# Versi lama: loop serial + img.verify() → sangat lambat di Drive (~1 jam+)
# Versi ini:  paralel 16 thread + img.load() → estimasi ~2–5 menit
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

MAX_WORKERS = 16  # naikkan ke 32 jika masih terasa lambat

def check_one(fp: str):
    """Return filepath jika corrupt, None jika OK."""
    try:
        with Image.open(fp) as img:
            img.load()   # load pixel — lebih cepat dari verify(), cukup untuk deteksi corrupt
        return None
    except Exception:
        return fp

filepaths = df['filepath'].tolist()
skip_list = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(check_one, fp): fp for fp in filepaths}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Cek corrupt'):
        result = future.result()
        if result is not None:
            skip_list.append(result)

print(f'\nGambar corrupt / tidak terbaca: {len(skip_list)}')
if skip_list:
    print('Contoh:')
    for fp in skip_list[:5]:
        print(f'  {fp}')

# Simpan skip-list
with open(OUTPUT_DIR / 'skip_list.txt', 'w') as f:
    f.write('\n'.join(skip_list))
print(f'Skip-list disimpan ke: {OUTPUT_DIR}/skip_list.txt')

# Remove dari df
df_clean = df[~df['filepath'].isin(set(skip_list))].reset_index(drop=True)
print(f'df setelah remove corrupt: {len(df_clean)} gambar')

eda_stats['corrupt_count'] = len(skip_list)
eda_stats['skip_list'] = skip_list

## Fase 1D — Cek Duplikat (Perceptual Hash)

NOTE:
> Ini bagian terlama (~15–30 menit untuk 26k gambar). Jalankan saat ada waktu.
> Hasil menentukan apakah perlu **group-aware split** di `02_split_verify.ipynb`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import imagehash

HASH_SIZE   = 8   # jangan diubah — konsisten dengan split notebook
MAX_WORKERS = 16  # sweet spot untuk Drive

def compute_phash_one(args):
    """Return (index, hash_str) — index wajib dikembalikan untuk menjaga urutan."""
    idx, fp = args
    try:
        with Image.open(fp) as img:
            return idx, str(imagehash.phash(img, hash_size=HASH_SIZE))
    except Exception:
        return idx, None   # gambar yang gagal dibaca → None

filepaths = df_clean['filepath'].tolist()
hashes = [None] * len(filepaths)   # ← pre-allocate, bukan append — urutan WAJIB dijaga

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(compute_phash_one, (idx, fp)): idx
               for idx, fp in enumerate(filepaths)}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Menghitung pHash'):
        idx, h = future.result()
        hashes[idx] = h   # ← tulis ke slot yang benar, bukan append acak

df_clean = df_clean.copy()
df_clean['phash'] = hashes
print(f'pHash dihitung untuk {df_clean["phash"].notna().sum()} gambar')
print(f'Gagal dibaca (None): {df_clean["phash"].isna().sum()} gambar')


In [ ]:
# Cek exact duplicates
hash_counts = df_clean['phash'].value_counts()
dup_hashes  = hash_counts[hash_counts > 1]

print(f'Hash unik yang muncul > 1 kali  : {len(dup_hashes)}')
print(f'Total gambar dalam grup duplikat : {dup_hashes.sum()}')

if len(dup_hashes) > 0:
    print('\nContoh grup duplikat:')
    for h, cnt in dup_hashes.head(3).items():
        members = df_clean[df_clean['phash'] == h]['filepath'].tolist()
        print(f'  Hash {h}: {cnt} gambar')
        for fp in members[:2]:
            print(f'    {fp}')

# Buat group_id — versi vektorisasi (jauh lebih cepat dari apply())
hash_to_group = {h: gid for gid, h in enumerate(dup_hashes.index)}

df_clean = df_clean.copy()
df_clean['group_id'] = df_clean['phash'].map(hash_to_group)   # NaN untuk non-duplikat

# Isi NaN (gambar unik) dengan ID unik yang tidak bentrok dengan grup duplikat
n_dup_groups   = len(hash_to_group)
mask_no_dup    = df_clean['group_id'].isna()
df_clean.loc[mask_no_dup, 'group_id'] = range(n_dup_groups, n_dup_groups + mask_no_dup.sum())
df_clean['group_id'] = df_clean['group_id'].astype(int)

print(f'\ngroup_id range: {df_clean["group_id"].min()} – {df_clean["group_id"].max()}')

eda_stats['duplicate_groups']       = len(dup_hashes)
eda_stats['duplicate_total_images'] = int(dup_hashes.sum()) if len(dup_hashes) > 0 else 0
eda_stats['need_group_aware_split'] = len(dup_hashes) > 0


## Fase 1E — Inspeksi Visual Sampel

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle('Sampel Gambar per Kelas (5 per kelas)', fontsize=14, fontweight='bold')

for row_idx, (label_idx, cls_name) in enumerate(zip([0, 1, 2], CLASS_NAMES)):
    samples = df_clean[df_clean['label'] == label_idx].sample(5, random_state=42)
    for col_idx, (_, record) in enumerate(samples.iterrows()):
        ax = axes[row_idx][col_idx]
        img = Image.open(record['filepath'])
        ax.imshow(img)
        ax.set_title(f'{cls_name}', fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print('Periksa: ada gambar aneh / mislabel?')

## Save Output Fase 1

In [ ]:
# Simpan df_clean dengan phash dan group_id untuk dipakai di notebook split
df_clean.to_csv(OUTPUT_DIR / 'df_clean.csv', index=False)
print(f'df_clean disimpan: {len(df_clean)} gambar')

# Simpan eda_stats
with open(OUTPUT_DIR / 'eda_stats.json', 'w') as f:
    json.dump(eda_stats, f, indent=2)
print(f'eda_stats disimpan')

print('\n=== RINGKASAN EDA ===')
print(json.dumps(eda_stats, indent=2))

In [ ]:
import shutil
from pathlib import Path

OUTPUT_DIR  = Path('/content/track_a_outputs')   # lokasi output yang sudah ada
DRIVE_OUTPUT = Path('/content/drive/MyDrive/BDC2026_TrackA_Outputs')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)

print('✅ Berhasil disalin ke Drive:')
for f in sorted(DRIVE_OUTPUT.iterdir()):
    print(f'   {f.name} ({f.stat().st_size:,} bytes)')


## Checklist Fase 0 & 1

Tandai setelah selesai:

- [ ] Dataset terunduh & jumlah terverifikasi (26.527 / 1.458)
- [ ] Distribusi kelas diketahui + kelas minoritas ditandai
- [ ] Statistik ukuran gambar diketahui → resolusi input ditetapkan
- [ ] Daftar gambar corrupt dibuat (skip-list)
- [ ] Cek duplikat selesai (ada/tidak, daftar grup)
- [ ] Inspeksi visual sampel per kelas

**Next:** Buka `02_split_verify.ipynb` untuk membuat `folds.csv` dan verifikasi dataloader.